# 04 · Logistic baseline vs gradient boosting

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

Now the prediction question: on **future** months, which model predicts lapses better? Train on months before 2025, test on 2025. Never shuffle time-ordered data into a random split: it lets the model peek at the future.

scikit-learn's `HistGradientBoostingClassifier` uses the same algorithm as LightGBM and needs no extra installation. LightGBM and XGBoost are worth knowing by name; the workflow is identical.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import sys
sys.path.insert(0, "../python")

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulate as sim
importlib.reload(sim)

members = pd.read_parquet("../data/members.parquet")
print(f"{len(members):,} members, {members['lapsed'].mean():.1%} lapsed")

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATS = list(sim.LEVELS)
CAT_LEVELS = [sim.LEVELS[c][0] for c in CATS]
NUMS = ["premium_increase", "tenure", "early", "anniversary", "long"]
FEATURE_COLS = CATS + NUMS
CUTOFF = "2025-01"   # train on months before this, test on this month onwards


def make_prep(dense=False):
    """One-hot the categories (reference level dropped) and scale the numbers."""
    return ColumnTransformer([
        ("cat", OneHotEncoder(categories=CAT_LEVELS, drop="first", sparse_output=not dense), CATS),
        ("num", StandardScaler(), NUMS),
    ])

pm = sim.person_months(members)
pm.head()

## 1. Split by time

### Your turn 1.1

Set `train` to the person-months with `calendar_month < CUTOFF` and `test` to those on or after it. Text months like `"2024-07"` sort correctly as strings, so a plain comparison works.

In [ ]:
train = ...
test = ...

In [ ]:
check("1.1 no overlap in time", lambda: train["calendar_month"].max() < test["calendar_month"].min())
check("1.1 every row used", lambda: len(train) + len(test), len(pm))

## 2. Logistic baseline

### Your turn 2.1

Build `lr`: a `Pipeline` of `make_prep()` and `LogisticRegression(max_iter=1000)`. Fit it on `train[FEATURE_COLS]` and `train["lapse"]`, then set `auc_lr` to its AUC on the test set.

In [ ]:
lr = ...
auc_lr = ...

In [ ]:
check("2.1 better than chance", lambda: auc_lr > 0.6)

## 3. Gradient boosting, tuned with time-ordered folds

`TimeSeriesSplit` makes folds that always train on earlier rows and validate on later ones. It works on row order, so sort the training data by `calendar_month` first.

### Your turn 3.1

1. `train_sorted = train.sort_values("calendar_month")`
2. `gbm`: a `Pipeline` of `make_prep(dense=True)` and `HistGradientBoostingClassifier(random_state=0)`
3. `search`: `GridSearchCV(gbm, {"model__learning_rate": [0.05, 0.1], "model__max_leaf_nodes": [15, 31]}, cv=TimeSeriesSplit(n_splits=3), scoring="roc_auc")`, fitted on the sorted training data. This takes a few minutes.
4. `auc_gbm`: the AUC of `search.best_estimator_` on the test set

In [ ]:
train_sorted = ...
gbm = ...
search = ...
auc_gbm = ...

In [ ]:
check("3.1 better than chance", lambda: auc_gbm > 0.6)
check("3.1 within 0.02 of the logistic model", lambda: abs(auc_gbm - auc_lr) < 0.02)

**Why boosting does not win here.** Your simulator is a linear model on exactly these features, so a well-specified logistic model is already close to the truth and there is nothing extra for trees to find. On real data, with interactions and non-linear effects, boosting usually pulls ahead. Saying this in an interview shows you understand *why* models win, not just which one did.

## 4. Save the test predictions for the evaluation notebook

In [ ]:
if hasattr(search, "best_estimator_") and hasattr(lr, "predict_proba"):
    preds = test[["member_id", "calendar_month", "tenure", "product_tier", "lapse"]].copy()
    preds["p_lr"] = lr.predict_proba(test[FEATURE_COLS])[:, 1]
    preds["p_gbm"] = search.best_estimator_.predict_proba(test[FEATURE_COLS])[:, 1]
    preds["train_rate"] = train["lapse"].mean()
    preds.to_parquet("../data/test_predictions.parquet", index=False)
    print(f"Saved {len(preds):,} test predictions")
else:
    print("Finish sections 2 and 3 first.")

## Wrap-up

Commit from VS Code: `Logistic baseline vs gradient boosting on a time split`. Add `data/test_predictions.parquet` to `.gitignore`.